In [1]:
import requests, pandas as pd, time, re, os
import urllib3
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup

urllib3.disable_warnings()

BASE_URL = "https://www.juntadeandalucia.es/institutodeestadisticaycartografia/intranet/admin/rest/v1.0"
HEADERS = {"Accept": "application/json", "User-Agent": "Mozilla/5.0"}
RUTA_BASE = r"C:\Users\JorgeB\Desktop\Proyecto-DA\badea_datos_sevilla"

secciones = {
    "1": "1_entorno_fisico_medio_ambiente",
    "2": "2_demografia_poblacion",
    "3": "3_sociedad",
    "4": "4_economia",
    "5": "5_mercado_trabajo",
    "6": "6_hacienda"
}

subsecciones = {
    "1.1": "1.1_Territorio", "1.2": "1.2_Medio_ambiente",
    "2.1": "2.1_Cifras_poblacion", "2.2": "2.2_Movimiento_natural",
    "2.3": "2.3_Migraciones", "2.4": "2.4_Indicadores_demograficos",
    "3.1": "3.1_Ensenanza", "3.2": "3.2_Recursos_sanitarios",
    "3.3": "3.3_Edificios_viviendas", "3.4": "3.4_Elecciones",
    "3.5": "3.5_Cultura", "3.6": "3.6_Servicios_sociales",
    "3.7": "3.7_Infraestructuras",
    "4.1": "4.1_Agricultura_ganaderia", "4.2": "4.2_Pesca",
    "4.3": "4.3_Energia", "4.4": "4.4_Turismo",
    "4.5": "4.5_Construccion_vivienda", "4.6": "4.6_Transporte",
    "4.7": "4.7_Comunicaciones", "4.8": "4.8_Inversiones",
    "4.9": "4.9_Sistema_financiero", "4.10": "4.10_Actividad_empresarial",
    "5.1": "5.1_Poblacion", "5.2": "5.2_Actividad",
    "5.3": "5.3_Empleo", "5.4": "5.4_Paro", "5.5": "5.5_Pensiones",
    "6.1": "6.1_Cuentas_administraciones", "6.2": "6.2_Estadisticas_catastrales",
    "6.3": "6.3_IAE", "6.4": "6.4_IRPF"
}

# 1. ÍNDICE
print("🔍 Obteniendo índice...")
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
driver.get("https://www.juntadeandalucia.es/institutodeestadisticaycartografia/badea/informe/anual?CodOper=b3_151&idNode=23204")
time.sleep(15)
soup = BeautifulSoup(driver.page_source, "html.parser")
driver.quit()

consultas = []
for a in soup.find_all("a", href=True):
    m = re.search(r"consulta/anual/(\d+)", a["href"])
    if m:
        consultas.append({"id": int(m.group(1)), "titulo": a.text.strip()})
consultas_unicas = list({c["id"]: c for c in consultas}.values())
print(f"✅ {len(consultas_unicas)} consultas")

# 2. SALTAR YA DESCARGADAS
ya_descargados = set()
for root, dirs, files in os.walk(RUTA_BASE):
    for f in files:
        if f.endswith(".parquet"):
            ya_descargados.add(int(f.split("_")[0]))
pendientes = [c for c in consultas_unicas if c['id'] not in ya_descargados]
print(f"⏭️ Ya descargadas: {len(ya_descargados)} | Pendientes: {len(pendientes)}")

# 3. FUNCIONES
def get_carpeta(titulo):
    m = re.match(r'^(\d+)\.(\d+)\.?', titulo)
    if m:
        sec, subsec = m.group(1), m.group(2)
        clave = f"{sec}.{subsec}"
        return os.path.join(RUTA_BASE, secciones.get(sec, f"{sec}_otros"), subsecciones.get(clave, clave))
    return os.path.join(RUTA_BASE, "otros")

def obtener_años(consulta_id):
    try:
        r = requests.get(f"{BASE_URL}/jerarquia/2?consultaId={consulta_id}&alias=D_TEMPORAL_0", headers=HEADERS, verify=False, timeout=30)
        return [{"id": h["id"], "cod": h["cod"]} for h in r.json()["data"]["children"]
                if h["cod"].isdigit() and 1990 <= int(h["cod"]) <= 2025]
    except:
        return []

def extraer_filas(data):
    """Filtra Sevilla por código exacto '41' o municipios '41XXX'. Si no hay Sevilla guarda todo."""
    medida = data["measures"][0]["des"]
    filas_sevilla = []
    filas_todas = []
    for fila in data["data"]:
        reg = {data["hierarchies"][j]["des"]: fila[j]["des"] for j in range(len(data["hierarchies"]))}
        reg[medida] = fila[-1]["val"]
        filas_todas.append(reg)
        es_sevilla = False
        for dim in fila[:-1]:
            for cod in dim.get("cod", []):
                if cod == "41" or (len(cod) == 5 and cod.startswith("41")):
                    es_sevilla = True
                    break
            if es_sevilla:
                break
        if es_sevilla:
            filas_sevilla.append(reg)
    return filas_sevilla if filas_sevilla else filas_todas

# 4. DESCARGA
errores = []
print(f"🚀 Descargando {len(pendientes)} consultas...\n")

for i, c in enumerate(pendientes):
    consulta_id, titulo = c['id'], c['titulo']
    print(f"[{i+1}/{len(pendientes)}] {consulta_id} - {titulo[:45]}", end=" → ")
    try:
        años = obtener_años(consulta_id)
        filas = []
        if años:
            for a in años:
                try:
                    r = requests.get(f"{BASE_URL}/consulta/{consulta_id}", params={"D_TEMPORAL_0": str(a["id"])}, headers=HEADERS, verify=False, timeout=30)
                    data = r.json()
                    if not data.get("data"): continue
                    filas += extraer_filas(data)
                except:
                    pass
                time.sleep(0.05)
        else:
            r = requests.get(f"{BASE_URL}/consulta/{consulta_id}", headers=HEADERS, verify=False, timeout=30)
            data = r.json()
            if data.get("data"):
                filas += extraer_filas(data)

        if not filas:
            print("⚠️ vacío")
            errores.append(c)
            continue

        df = pd.DataFrame(filas)
        carpeta = get_carpeta(titulo)
        os.makedirs(carpeta, exist_ok=True)
        nombre = f"{consulta_id}_{titulo[:40].replace(' ','_').replace('/','_').replace('.','')}.parquet"
        df.to_parquet(os.path.join(carpeta, nombre), index=False)
        print(f"✅ {len(df)} filas")

    except Exception as e:
        print(f"❌ {e}")
        errores.append(c)
    time.sleep(0.1)

print(f"\n🏁 Sesión terminada. Errores: {len(errores)}")
print(f"📦 Total descargadas: {len(ya_descargados) + len(pendientes) - len(errores)}")

🔍 Obteniendo índice...
✅ 454 consultas
⏭️ Ya descargadas: 299 | Pendientes: 155
🚀 Descargando 155 consultas...

[1/155] 19607 - 3.4.1.1. Elecciones Generales: Censo electora → ✅ 1176 filas
[2/155] 19608 - 3.4.1.2. Elecciones Generales: Número de vota → ✅ 1176 filas
[3/155] 19609 - 3.4.1.3. Elecciones Generales: Votos y absten → ✅ 4704 filas
[4/155] 19610 - 3.4.1.4. Elecciones Generales: Votos a candid → ✅ 7377 filas
[5/155] 101735 - 3.4.4.4. Elecciones Europeas en Andalucía:  V → ⚠️ vacío
[6/155] 19946 - 3.7.7. Residuos urbanos:Contenedores y produc → ✅ 42778 filas
[7/155] 19947 - 3.7.8. Alumbrado público:Número de puntos de  → ✅ 5428 filas
[8/155] 29300 - 4.1.3.16. Ganadería: Número de explotaciones → ✅ 840 filas
[9/155] 29301 - 4.1.3.17. Ganadería:Número de cabezas → ✅ 840 filas
[10/155] 29302 - 4.1.3.18. Ganadería: Unidades ganaderas → ✅ 735 filas
[11/155] 1960 - 4.1.4.4. Superficie de las explotaciones agra → ⚠️ vacío
[12/155] 117647 - 4.4.1.2.1. Hoteles: establecimientos y plazas 